# 03 – Bivariate Analysis and Conclusions

**Objective:** analyse how `log(PRICE)` relates to each candidate feature, identify the most predictive variables, and draw conclusions for the modelling stage.

**Input:** `data/processed/listings_01.parquet` (output of notebook 01).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# Consistent visual style
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

PERIOD_MAP = {"201803": "Q1 2018", "201806": "Q2 2018", "201809": "Q3 2018", "201812": "Q4 2018"}

try:
    df = pd.read_parquet("../data/processed/listings_01.parquet")
except FileNotFoundError:
    df = pd.read_csv("../data/Madrid_Sale.csv")

df["log_price"] = np.log(df["PRICE"])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"log(PRICE) — mean: {df['log_price'].mean():.2f}, std: {df['log_price'].std():.2f}")

## 1. Continuous features vs log(PRICE)

In [ ]:
cont_features = ["CONSTRUCTEDAREA", "CADCONSTRUCTIONYEAR", "DISTANCE_TO_CITY_CENTER", "DISTANCE_TO_METRO"]

readable = {
    "CONSTRUCTEDAREA": "Constructed area (m²)",
    "CADCONSTRUCTIONYEAR": "Construction year",
    "DISTANCE_TO_CITY_CENTER": "Distance to city centre (km)",
    "DISTANCE_TO_METRO": "Distance to nearest metro (km)",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.ravel(), cont_features):
    ax.scatter(df[feat], df["log_price"], alpha=0.02, s=1, color="#2c3e50")
    z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), "log_price"], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), color="#e74c3c", lw=2)
    r = df[feat].corr(df["log_price"])
    ax.set_title(f"{readable.get(feat, feat)} (r = {r:.3f})")
    ax.set_ylabel("log(Price)")
    ax.set_xlabel(readable.get(feat, feat))
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Continuous features vs log(Price)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

- **`CONSTRUCTEDAREA`** has the strongest linear relationship with log(PRICE) — larger flats are more expensive, as expected. This will likely be the single most predictive feature.
- **`DISTANCE_TO_CITY_CENTER`** shows a clear negative correlation: properties closer to the centre command higher prices.
- **`CADCONSTRUCTIONYEAR`** shows a mild positive trend — newer buildings tend to be pricier, but with high variance.
- **`DISTANCE_TO_METRO`** has a weaker negative correlation than distance to the centre, but still contributes location information.

## 2. Discrete and ordinal features vs log(PRICE)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ROOMNUMBER
room_data = df.copy()
room_data["ROOMNUMBER_grp"] = room_data["ROOMNUMBER"].clip(upper=7).astype(str)
room_data.loc[room_data["ROOMNUMBER"] >= 7, "ROOMNUMBER_grp"] = "7+"
sns.boxplot(x="ROOMNUMBER_grp", y="log_price", data=room_data, ax=axes[0, 0],
            order=[str(i) for i in range(7)] + ["7+"], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[0, 0].set_title("Number of rooms")
axes[0, 0].set_xlabel("Rooms")

# BATHNUMBER
bath_data = df.copy()
bath_data["BATHNUMBER_grp"] = bath_data["BATHNUMBER"].clip(upper=4).astype(str)
bath_data.loc[bath_data["BATHNUMBER"] >= 4, "BATHNUMBER_grp"] = "4+"
sns.boxplot(x="BATHNUMBER_grp", y="log_price", data=bath_data, ax=axes[0, 1],
            order=[str(i) for i in range(4)] + ["4+"], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[0, 1].set_title("Number of bathrooms")
axes[0, 1].set_xlabel("Bathrooms")

# AMENITYID
amenity_labels = {1: "Basic", 2: "Standard", 3: "Premium"}
amenity_data = df.copy()
amenity_data["AMENITYID_lbl"] = amenity_data["AMENITYID"].map(amenity_labels)
sns.boxplot(x="AMENITYID_lbl", y="log_price", data=amenity_data, ax=axes[0, 2],
            order=["Basic", "Standard", "Premium"], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[0, 2].set_title("Amenity level")
axes[0, 2].set_xlabel("")

# CADASTRALQUALITYID
sns.boxplot(x="CADASTRALQUALITYID", y="log_price", data=df, ax=axes[1, 0], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1, 0].set_title("Cadastral quality (0 = best, 9 = worst)")
axes[1, 0].set_xlabel("Quality level")

# FLOORCLEAN
floor_data = df.copy()
floor_data["FLOOR_grp"] = floor_data["FLOORCLEAN"].clip(-1, 8).astype(int).astype(str)
floor_data.loc[floor_data["FLOORCLEAN"] > 8, "FLOOR_grp"] = "9+"
sns.boxplot(x="FLOOR_grp", y="log_price", data=floor_data, ax=axes[1, 1],
            order=[str(i) for i in range(-1, 9)] + ["9+"], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1, 1].set_title("Floor number")
axes[1, 1].set_xlabel("Floor")

# FLATLOCATIONID
loc_labels = {0: "Unknown", 1: "Interior", 2: "Exterior"}
loc_data = df.copy()
loc_data["LOC_lbl"] = loc_data["FLATLOCATIONID"].map(loc_labels)
sns.boxplot(x="LOC_lbl", y="log_price", data=loc_data, ax=axes[1, 2],
            order=["Unknown", "Interior", "Exterior"], color="#3498db",
            flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1, 2].set_title("Flat location")
axes[1, 2].set_xlabel("")

for ax in axes.ravel():
    ax.set_ylabel("log(Price)")
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Discrete features vs log(Price)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

- **`ROOMNUMBER`** and **`BATHNUMBER`**: price increases with room/bath count, but flattens or drops for very high values (small sample sizes).
- **`CADASTRALQUALITYID`**: strong monotonic relationship — quality 0 (best) has the highest prices, quality 9 (worst) the lowest. This is one of the most informative features.
- **`AMENITYID`**: clear ordinal effect — higher amenity levels correspond to higher prices.
- **`FLOORCLEAN`**: higher floors tend to cost more. Basement (-1) has the lowest median price.
- **`FLATLOCATIONID`**: exterior flats (2) are more expensive than interior ones (1), as expected. Unknown (0) falls in between.

## 3. Binary features vs log(PRICE)

In [ ]:
binary_cols = [c for c in df.columns if c.startswith("HAS") or c.startswith("IS") or c.startswith("BUILTTYPEID")]

price_diff = {}
for col in binary_cols:
    median_0 = df.loc[df[col] == 0, "PRICE"].median()
    median_1 = df.loc[df[col] == 1, "PRICE"].median()
    price_diff[col] = ((median_1 - median_0) / median_0) * 100

price_diff = pd.Series(price_diff).sort_values()

readable = {
    "HASTERRACE": "Terrace", "HASLIFT": "Lift", "HASAIRCONDITIONING": "Air conditioning",
    "HASPARKINGSPACE": "Parking", "ISPARKINGSPACEINCLUDEDINPRICE": "Parking incl. in price",
    "HASNORTHORIENTATION": "North facing", "HASSOUTHORIENTATION": "South facing",
    "HASEASTORIENTATION": "East facing", "HASWESTORIENTATION": "West facing",
    "HASBOXROOM": "Box room", "HASWARDROBE": "Wardrobe", "HASSWIMMINGPOOL": "Swimming pool",
    "HASDOORMAN": "Doorman", "HASGARDEN": "Garden", "ISDUPLEX": "Duplex",
    "ISSTUDIO": "Studio", "ISINTOPFLOOR": "Top floor",
    "BUILTTYPEID_1": "Built type 1", "BUILTTYPEID_2": "Built type 2", "BUILTTYPEID_3": "Built type 3",
}
labels = [readable.get(c, c) for c in price_diff.index]

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#c0392c" if v < 0 else "#27ae60" for v in price_diff]
ax.barh(labels, price_diff.values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Median price difference when feature = 1 (%)")
ax.set_title("Impact of binary features on median sale price")
ax.axvline(0, color="black", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(price_diff.round(1))

- **`HASSWIMMINGPOOL`**, **`HASAIRCONDITIONING`**, **`HASLIFT`**, and **`HASDOORMAN`** are associated with the largest price premiums — these are markers of higher-quality buildings.
- **`ISSTUDIO`** is associated with a large price *discount*, as expected (studios are small and cheap).
- **`HASTERRACE`**, **`HASBOXROOM`**, and **`HASWARDROBE`** have moderate positive effects.
- Orientation features (`HASNORTH/SOUTH/EAST/WESTORIENTATION`) show modest differences.

## 4. Correlation matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = ["log_price"]
numeric_cols = [c for c in numeric_cols if c not in exclude]

corr = df[numeric_cols].corr()

# Sort by correlation with PRICE
price_corr = corr["PRICE"].drop("PRICE").sort_values(ascending=False)
ordered_cols = ["PRICE"] + price_corr.index.tolist()

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(corr.loc[ordered_cols, ordered_cols], annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax, square=True,
            annot_kws={"size": 6}, vmin=-1, vmax=1,
            linewidths=0.3, linecolor="white")
ax.set_title("Correlation matrix (ordered by correlation with PRICE)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

print("Top 10 features by |correlation| with PRICE:")
print(price_corr.abs().sort_values(ascending=False).head(10).round(3))

The correlation matrix confirms the bivariate findings:
- **`CONSTRUCTEDAREA`** has the highest correlation with PRICE, followed by size-related features (`BATHNUMBER`, `ROOMNUMBER`).
- **Distance features** are negatively correlated — closer to the centre means higher price.
- **`CADASTRALQUALITYID`** is negatively correlated (higher quality index = worse quality = lower price).
- Among binary features, `HASSWIMMINGPOOL`, `HASLIFT`, and `HASAIRCONDITIONING` stand out.
- Some features are highly correlated with each other (e.g., `DISTANCE_TO_CITY_CENTER` and `DISTANCE_TO_CASTELLANA`), suggesting potential multicollinearity to monitor during modelling.

## 5. Geographic analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(df["LONGITUDE"], df["LATITUDE"],
                     c=df["log_price"], cmap="RdYlGn",
                     alpha=0.2, s=2, vmin=df["log_price"].quantile(0.05),
                     vmax=df["log_price"].quantile(0.95))
cbar = plt.colorbar(scatter, ax=ax, label="log(Price)", shrink=0.7)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Sale prices across Madrid — clear centre-to-periphery gradient", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The map reveals a clear **price gradient from the centre outward**: the most expensive properties cluster in the Salamanca, Chamberí, and Centro districts (centre-north), while the southern and peripheral neighbourhoods have significantly lower prices. The Paseo de la Castellana axis (running north-south through the city centre) is clearly visible as a high-price corridor, which explains why `DISTANCE_TO_CASTELLANA` is a useful feature.

## 6. Price trend over 2018

In [ ]:
period_col = "PERIOD"
if df[period_col].dtype != object:
    df[period_col] = df[period_col].astype(str)

period_stats = df.groupby(period_col)["PRICE"].agg(["median", "mean", "count"])
period_stats.index = [PERIOD_MAP.get(p, p) for p in period_stats.index]
print(period_stats)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(period_stats.index, period_stats["median"] / 1000, marker="o", linewidth=2.5,
        markersize=8, color="#2c3e50")
ax.fill_between(period_stats.index, period_stats["median"] / 1000 * 0.98,
                period_stats["median"] / 1000 * 1.02, alpha=0.1, color="#2c3e50")
ax.set_ylabel("Median price (thousands EUR)")
ax.set_title("Median sale price by quarter — stable through 2018")
ax.set_xlabel("")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Prices are relatively stable across the four 2018 quarters, with a mild upward trend. The temporal effect is small compared to the cross-sectional variation, so `PERIOD` can be included as a control variable but is unlikely to be a major predictor.

## 7. Conclusions

### Most predictive features (by bivariate analysis)

| Rank | Feature | Relationship | Strength |
|---|---|---|---|
| 1 | `CONSTRUCTEDAREA` | Positive, linear | Very strong |
| 2 | `CADASTRALQUALITYID` | Negative (higher = worse quality) | Strong |
| 3 | `DISTANCE_TO_CITY_CENTER` | Negative | Strong |
| 4 | `BATHNUMBER` | Positive | Moderate–strong |
| 5 | `LATITUDE` / `LONGITUDE` | Spatial gradient | Strong |
| 6 | `HASSWIMMINGPOOL`, `HASLIFT`, `HASAIRCONDITIONING` | Positive | Moderate |
| 7 | `DISTANCE_TO_CASTELLANA` | Negative | Moderate |
| 8 | `ROOMNUMBER` | Positive | Moderate |
| 9 | `AMENITYID` | Positive (ordinal) | Moderate |
| 10 | `FLOORCLEAN` | Positive | Weak–moderate |

### Recommended feature set for modelling

**Include:** `CONSTRUCTEDAREA`, `ROOMNUMBER`, `BATHNUMBER`, `FLOORCLEAN`, `FLATLOCATIONID`, `CADCONSTRUCTIONYEAR`, `CADASTRALQUALITYID`, `CADMAXBUILDINGFLOOR`, `CADDWELLINGCOUNT`, `AMENITYID`, all `HAS*`/`IS*` binary features, `BUILTTYPEID_1/2/3`, `DISTANCE_TO_CITY_CENTER`, `DISTANCE_TO_METRO`, `DISTANCE_TO_CASTELLANA`, `LATITUDE`, `LONGITUDE`, `PERIOD`.

**Exclude:**
- `UNITPRICE` — target leakage (already dropped in notebook 01).
- `ASSETID` — identifier with no predictive value.
- `CONSTRUCTIONYEAR` — dropped in notebook 01 (59% missing, replaced by `CADCONSTRUCTIONYEAR`).
- `PARKINGSPACEPRICE` — inconsistent data; parking information is captured by `HASPARKINGSPACE` and `ISPARKINGSPACEINCLUDEDINPRICE`.

### Modelling recommendations

1. **Log-transform the target:** `log(PRICE)` is approximately normal and linearises relationships with area and distance features.
2. **Watch for multicollinearity:** `DISTANCE_TO_CITY_CENTER` and `DISTANCE_TO_CASTELLANA` are highly correlated; similarly, `ROOMNUMBER` and `CONSTRUCTEDAREA`. Consider regularisation or dropping one of each pair.
3. **Spatial features are key:** latitude, longitude, and distance features together capture the dominant price gradient. A model that ignores location will perform poorly.
4. **Building quality matters:** `CADASTRALQUALITYID` and amenity-related binaries capture building-level effects that go beyond the flat's physical characteristics.